# 科创人才历史客户授信额度组合优化

> **必要前置步骤：**先将任务2 Notebook 完整运行两次：一次 `TARGET='y_freq'`，一次 `TARGET='y_dq_risk'`。读取两个 `sampling_method_selection_*.csv` 中 `selected=True` 的结果并填入本 Notebook；缺少任一目标结果时不要继续生成概率网格。

**运行顺序**
1. 固定数据、模型、采样、校准、风险收益和业务约束参数；
2. 读取、清洗并生成历史客户双标签；
3. 采样方法固定后使用训练集+验证集建模、独立校准集拟合概率校准器，并先完成参数估计与候选实测；
4. 对全部历史客户生成折外校准概率，并执行一次联合组合优化；
5. 输出模型目标函数变化、额度调整、预算使用、人才等级、概率变化、边界与集中度报告。

历史数据无法观察建议额度下的反事实结果，Notebook 中的目标函数变化仅代表当前模型、参数和约束下的离线变化，不能直接解释为实际利润提升。


---
## Cell 1 · 参数配置
**所有参数统一在这里修改，其他 Cell 直接引用。**

In [ ]:
# ====================================================================
# ★ 参数配置区 ★  ← 每次跑前只需改这里（对齐 kechuang_potential_preprocessing.ipynb）
# ====================================================================

# ── 数据文件 ──────────────────────────────────────────────────────────
DATA_FILE     = "kechuang_merged0729.csv"   # 与任务2使用同一原始数据（支持 .csv / .xlsx）
CSV_ENCODING  = "utf-8-sig"                             # CSV 编码；乱码可改 "gbk"

# ── 清洗后数据保存路径（Cell 3 使用；若跳过清洗则直接读此文件）────────
CLEANED_FILE = "data_cleaned.csv"

# ── 是否跳过清洗 Cell（True = 直接用 CLEANED_FILE，False = 重新清洗）──
SKIP_CLEANING = False

# ── 快照日期（用于 days_since_become_cust 等衍生特征）────────────────
SNAPSHOT_DATE = "2026-06-24"

# ── y_freq 频繁支用因变量构造模式（参照 load_kechuang_potential_data.py）──
# "bout_gt0_and_curr_p80" : ba_out_bal_diff > 0  且 ac_curr_bal_diff >= P80（默认）
# "bout_p80_and_accr_p80" : ba_out_bal_diff >= P80  且 ac_accr_bal_diff >= P80
# "curr_p80_only"         : ac_curr_bal_diff >= P80（单条件）
# "curr_p80_and_bout_p80" : ac_curr_bal_diff >= P80  且 ba_out_bal_diff >= P80
Y_FREQ_MODE = "curr_p80_and_bout_p80"

# ── 到期日筛选开关 ───────────────────────────────────────────────────
APPLY_MATURITY_FILTER = False          # True=筛选；False=跳过（当前默认关闭）
MATURITY_CUTOFF       = "2026-07-21"   # 与任务2保持一致；筛选关闭时不生效

# ── 贷款生效日筛选（聚合前，确保 X 特征表有对应快照覆盖）────────────
APPLY_EFF_DATE_FILTER = False
EFF_DATE_LOWER        = "2025-01-01"   # 与任务2保持一致
EFF_DATE_UPPER        = "2026-03-31"   
DEDUP_CST_LOAN        = False          # 默认仅报告重复；True 仅删除字段完全一致的重复账户，冲突重复会报错

# ── 授信额度多项式特征（概率网格训练时不加 sq/cube，此处供清洗阶段参考）──
ADD_QUOTA_SQ   = False
ADD_QUOTA_CUBE = False
ADD_QUOTA_LOG  = False

# ── LightGBM 建模参数（Cell 5 训练支用/违约双模型时使用）──────────────
RANDOM_STATE          = 42
HANDLE_IMBALANCE      = False    # True = 启用 scale_pos_weight（负样本数/正样本数）
EARLY_STOPPING_ROUNDS = 50
LGB_PARAMS = {
    "objective":         "binary",
    "metric":            "auc",
    "n_estimators":      500,
    "learning_rate":     0.05,
    "num_leaves":        31,
    "max_depth":         -1,
    "min_child_samples": 20,
    "subsample":         0.8,
    "bagging_freq":      1,
    "colsample_bytree":  0.8,
    "reg_alpha":         0.1,
    "reg_lambda":        0.1,
    "random_state":      42,
    "verbose":           -1,
}

# ── 全量应用：五折交叉拟合参数 ────────────────────────────
CROSS_FIT_FOLDS = 5
INNER_CROSS_FIT_FOLDS = 5  # 每个外层建模折内部再次交叉拟合，产生校准器训练所需的内部折外概率
SPLIT_RATIOS = (0.60, 0.15, 0.15, 0.10)  # 训练/验证/校准/最终测试

# 【必要前置步骤】先将任务2 Notebook 完整运行两次：
#   ① TARGET='y_freq'    -> sampling_method_selection_y_freq.csv
#   ② TARGET='y_dq_risk' -> sampling_method_selection_y_dq_risk.csv
# 分别读取两个文件中 selected=True 行的 method_key，并同步对应比例/集成参数。
# 若 selected 方法为 baseline，则这里对应的 SAMPLING_METHOD_* 保持 None。
# 分别填入任务2针对 y_freq / y_dq_risk 最终选定的采样方法；None 表示无采样。
# 可选: random_over / smote / borderline_smote / adasyn / random_under / tomek / enn /
#       smoteenn / smotetomek / balance_cascade / easy_ensemble。
# 两个目标必须分别与任务2对应目标的最终选择保持一致。
SAMPLING_METHOD_USAGE = "balance_cascade"       # 任务2 TARGET='y_freq' 的最终方法
SAMPLING_STRATEGY_USAGE = 1.0
SAMPLING_N_ESTIMATORS_USAGE = 10
SAMPLING_ENSEMBLE_RATIO_USAGE = 1.0
SAMPLING_METHOD_DEFAULT = "smote"     # 任务2 TARGET='y_dq_risk' 的最终方法
SAMPLING_STRATEGY_DEFAULT = 1.0
SAMPLING_N_ESTIMATORS_DEFAULT = 10
SAMPLING_ENSEMBLE_RATIO_DEFAULT = 1.0
SAMPLING_CATEGORICAL_CANDIDATES = [
    "gnd_cd", "mar_sttn_cd", "education_cd", "occup_cd", "cst_star_cd",
    "busikind",
]

# ── 额度网格参数 ────────────────────────────────────────────────────
# 原始授信额度达到或超过该阈值的客户，不进入标签、建模、参数估计或额度优化。
MAX_ORIGINAL_CREDIT_LIMIT = 1_000_000.0
GRID_MIN     = 1000.0      # 待选额度区间最小值（元）
GRID_MAX     = 1_000_000.0   # 待选额度区间最大值（元）
GRID_STEP    = 500.0       # 步长（元）
INCLUDE_ZERO = True        # True = 额外加入「0额度」点（拒贷选项）

# ── 风险调整收益与组合约束参数 ──────────────────────────────────────
# pi_i(L)=r_i*L*u_i-lambda_LGD*L*p_default_i(L)*u_i-c1*L-c2*L^2
# u_i 为客户历史支用率；频繁支用概率网格不再进入目标函数。
# 人才等级不乘入收益，只通过等级额度区间和组均额度单调约束体现。
INTEREST_RATE = 0.03       # 收益项系数 r
FTP_RATE = 0.02            # LINEAR_COST_MODE="ftp_utilization" 时使用
# "manual"：c1 使用下面的手工常数；"ftp_utilization"：c1_i=FTP_RATE×u_i
LINEAR_COST_MODE = "manual"
LINEAR_COST_MANUAL_VALUE = 0.0  # 当前按要求将 c1 设为0
UTILIZED_BALANCE_DIFF_COLUMN = "ac_curr_bal_diff"  # 口径1：期末累计支用-期初累计支用
UTILIZED_BALANCE_END_COLUMN = "ac_curr_bal_end"    # 口径2：期末累计支用
# "end"=期末累计支用合计/授信额度；"difference"=期初期末差值合计/授信额度
UTILIZATION_METHOD = "end"  # 当前默认选择第二种口径进入额度优化
CAP_UTILIZATION_ABOVE_100 = False  # True=将所选口径中>100%的客户支用率调整为100%
UTILIZATION_LOOKUP_FILE = "utilization_customer_lookup.csv"
UTILIZATION_DISTRIBUTION_FILE = "utilization_distribution.csv"
UTILIZATION_STATISTICS_FILE = "utilization_statistics.csv"
LGD_HISTORY_FILE = None    # 有可靠逐期回收明细时填 CSV/XLSX；否则显式使用下方情景
# 目标函数风险项系数 lambda_LGD；当前手工指定为0.3。
LGD_SELECTED_SCENARIO = "manual_0_3"
LGD_SCENARIOS = {
    "manual_0_3": 0.3,
    "optimistic": 0.5,
    "neutral": 0.7,
    "conservative": 0.9,
}
LGD_COLUMN_MAP = {
    "loan_id_col": "loanacctno",
    "observation_date_col": "observation_date",
    "account_status_col": "account_status",
    "outstanding_principal_col": "outstanding_principal",
    "cumulative_principal_paid_col": "cumulative_principal_paid",
    "close_date_col": "close_date",
    "cutoff_date": SNAPSHOT_DATE,
}
C2_REFERENCE_LIMIT = 1_100_000.0
C2_DELTAS = (0.0, 0.001, 0.0025, 0.005)
C2_CANDIDATES = (0.0,)  # 测试阶段当前只保留 c2=0；单元素元组的逗号不能省略
# "manual"=全量数据使用手工值；"selected"=使用上面候选集合选出的c2。
FULL_DATA_C2_MODE = "manual"
FULL_DATA_C2_MANUAL_VALUE = 0.0  # 当前按要求将全量应用c2设为0
C2_MAX_ZERO_RATE = None       # 若业务给出不可接受阈值，在此填写 0~1
C2_MAX_UPPER_HIT_RATE = None  # 若业务给出不可接受阈值，在此填写 0~1
C2_CLOSE_RELATIVE_TOLERANCE = 0.01  # 净收益接近时选更小 c2
# 客户优化额度下限=max(客户原始额度-customer_min_limit_decrease, 所属人才等级最低额度)。
# 等级最低额度：F3/F2/F1=1000元，E=20000元，D=100000元；A/B/C暂为0元。
CUSTOMER_MIN_LIMIT_ENABLED = True
CUSTOMER_MIN_LIMIT_DECREASE = 10_000.0
TIER_MIN_LIMIT_POLICY = {1: 1_000.0, 2: 1_000.0, 3: 1_000.0, 4: 20_000.0, 5: 100_000.0, 6: 0.0, 7: 0.0, 8: 0.0}
TIER_SHRINK_K = 5000.0
TIER_OVERALL_QUANTILE = 0.99
GROUP_MEAN_MIN_RATIO = {2: 1.0, 3: 1.0, 4: 0.8, 5: 0.8}
RISK_TOLERANCE = 1.05
LGD_COEFFICIENT = None  # 由参数估计 Cell 写入
LINEAR_COST = None      # 由参数估计 Cell 写入
QUADRATIC_COST = None   # 由 c2 候选实测 Cell 写入
TIER_MIN_LIMITS = None  # 由参数估计 Cell 写入
TIER_MAX_LIMITS = None  # 由参数估计 Cell 写入

# None 表示：总额度预算=历史总额度；风险预算=1.05×历史额度加权校准违约风险。
TOTAL_BUDGET = None
RISK_BUDGET = None
ENFORCE_GROUP_MEAN_MONOTONIC = True

# Python 3.6 / SciPy 1.1 默认使用拉格朗日引导近似解；同时提供快速贪心入口。
# 两种近似后端都检查全部硬约束，但不保证全局最优，也不产生 MIP gap。
SOLVER_BACKEND = "lagrangian"  # lagrangian / heuristic
OPTIMIZER_MAX_VARIABLES = 400_000
OPTIMIZER_CANDIDATES_PER_CUSTOMER = 16
HEURISTIC_MAX_ROUNDS = 30
LAGRANGIAN_ITERATIONS = 40             # 向量化定价迭代上限；速度优先可降至20
LAGRANGIAN_TIME_LIMIT_SECONDS = 60.0   # 仅限制拉格朗日定价循环
LAGRANGIAN_STEP_SIZE = 0.5
LAGRANGIAN_MULTIPLIER_CAP = 1000.0
LOCAL_SEARCH_ENABLED = True             # 仅 lagrangian；False 可跳过搜索
LOCAL_SEARCH_MAX_PASSES = 3             # 单客户双向调整 + 两客户交换轮数
LOCAL_SEARCH_TIME_LIMIT_SECONDS = 30.0  # 独立于拉格朗日定价时限
LOCAL_SEARCH_PAIR_CANDIDATE_POOL = 80   # 交换搜索候选池；越小越快
MILP_TIME_LIMIT_SECONDS = 600.0  # 仅 scipy_milp 后端使用
MILP_RELATIVE_GAP = 0.01         # 仅 scipy_milp 后端使用

# ── 额度优化/概率网格客户抽样开关（不缩减模型训练与校准数据）─────
# 开启后，在 Cell 5 前固定 fit/test/all 三个范围的人才等级分层抽样名单。
# 模型训练与 Isotonic 校准仍使用完整配置范围，只为抽样客户扫描候选额度网格；
# Cell 6B/7/8 直接复用对应名单，不再次抽样。
OPTIMIZATION_SAMPLE_ENABLED = False
OPTIMIZATION_SAMPLE_SIZE = 5000
OPTIMIZATION_SAMPLE_RANDOM_STATE = 42

# ── 耗时步骤复用开关 ─────────────────────────────────────
# 首次或数据/模型/网格参数变更时必须设 False 重跑。
# 完整运行并保存后，下次可设 True 直接复用。
REUSE_PROBABILITY_GRID = True    # 本次仅替换目标函数支用项，可直接复用原概率网格
REUSE_OPTIMIZATION = False       # True=跳过对应组合优化，恢复同参数/同后端的结果和状态

# ── 概率网格 & 结果目录 ─────────────────────────────────────────────
PROB_GRID_BASE_DIR = "probability_grid_large"
if OPTIMIZATION_SAMPLE_ENABLED:
    _PROB_GRID_SAMPLE_TAG = (
        f"_sampled_n{int(OPTIMIZATION_SAMPLE_SIZE)}"
        f"_rs{int(OPTIMIZATION_SAMPLE_RANDOM_STATE)}"
    )
else:
    _PROB_GRID_SAMPLE_TAG = ""
PROB_GRID_DIR = PROB_GRID_BASE_DIR + _PROB_GRID_SAMPLE_TAG
DEV_GRID_DIR = PROB_GRID_DIR + "_dev_calibrated"
CROSSFIT_GRID_DIR = PROB_GRID_DIR + "_crossfit_calibrated"
REPORTS_BASE_DIR = "reports"
REPORTS_DIR = REPORTS_BASE_DIR + _PROB_GRID_SAMPLE_TAG
PARAMETER_REPORTS_DIR = REPORTS_DIR + "/parameter_selection"
TEST_REPORTS_DIR = REPORTS_DIR + "/test_offline_evaluation"
FULL_REPORTS_DIR = REPORTS_DIR + "/full_crossfit_optimization"
VIZ_DIR = FULL_REPORTS_DIR + "/viz"

print("✓ 参数配置完成")
print(f"  数据文件          : {DATA_FILE}")
print(f"  清洗后文件        : {CLEANED_FILE}  (SKIP_CLEANING={SKIP_CLEANING})")
print(f"  快照日期          : {SNAPSHOT_DATE}")
print(f"  y_freq 构造模式   : {Y_FREQ_MODE}")
print(f"  到期日筛选        : {'开启 cutoff=' + MATURITY_CUTOFF if APPLY_MATURITY_FILTER else '关闭'}")
print(f"  生效日筛选        : {'开启 ' + EFF_DATE_LOWER + ' ~ ' + EFF_DATE_UPPER if APPLY_EFF_DATE_FILTER else '关闭'}")
print(f"  LightGBM 不平衡   : HANDLE_IMBALANCE={HANDLE_IMBALANCE}")
print(f"  额度网格          : {'[0] ∪ ' if INCLUDE_ZERO and GRID_MIN > 0 else ''}[{GRID_MIN:,.0f}, {GRID_MAX:,.0f}] 步长={GRID_STEP:.0f}")
print(f"  数据划分          : train/validation/cal/test={SPLIT_RATIOS}")
print(f"  支用率优化口径    : {UTILIZATION_METHOD}（end=期末累计支用；difference=期初期末差值）")
print("  待估计参数        : LGD、逐客户c1_i=FTP×u_i、等级上限；c2 将比较 4 个候选值")
print(f"  交叉拟合        : 外层={CROSS_FIT_FOLDS}折，内部校准交叉拟合={INNER_CROSS_FIT_FOLDS}折")
print(f"  y_freq 固定采样 : {SAMPLING_METHOD_USAGE or '无采样'}; strategy={SAMPLING_STRATEGY_USAGE}, members={SAMPLING_N_ESTIMATORS_USAGE}, ratio={SAMPLING_ENSEMBLE_RATIO_USAGE}")
print(f"  y_dq 固定采样   : {SAMPLING_METHOD_DEFAULT or '无采样'}; strategy={SAMPLING_STRATEGY_DEFAULT}, members={SAMPLING_N_ESTIMATORS_DEFAULT}, ratio={SAMPLING_ENSEMBLE_RATIO_DEFAULT}")
print(f"  优化抽样        : {'开启，样本数=' + str(OPTIMIZATION_SAMPLE_SIZE) if OPTIMIZATION_SAMPLE_ENABLED else '关闭'}")
print(f"  当前概率网格目录: {PROB_GRID_DIR}（抽样开关与优化范围同步）")
print(f"  当前结果目录    : {REPORTS_DIR}（抽样与全量互不覆盖）")


---
## Cell 2 · 读取原始数据 & 基础检查

In [ ]:
import os, sys
import numpy as np
import pandas as pd

# 确保辅助模块在路径中
sys.path.insert(0, os.getcwd())

if DATA_FILE.lower().endswith(".csv"):
    df_raw = pd.read_csv(DATA_FILE, encoding=CSV_ENCODING)
else:
    df_raw = pd.read_excel(DATA_FILE)

# 旧版中文列名兼容
if "授信额度" in df_raw.columns and "credamt" not in df_raw.columns:
    df_raw = df_raw.rename(columns={"授信额度": "credamt"})
    print("  字段重命名: 授信额度 → credamt")
if "dq_hist_dat_ctr" in df_raw.columns:
    df_raw = df_raw.rename(columns={"dq_hist_dat_ctr": "dq_hist_day_ctr"})
    print("  字段重命名: dq_hist_dat_ctr → dq_hist_day_ctr")

print(f"原始数据: {len(df_raw):,} 行 × {len(df_raw.columns)} 列")
print(f"列名预览: {list(df_raw.columns[:10])} ...")
df_raw.head(3)

---
## Cell 3 · 数据清洗
**若 `SKIP_CLEANING = True`，本 Cell 会自动跳过，直接读取 `CLEANED_FILE`。**

清洗流程（对齐 `load_kechuang_potential_data.py` + `kechuang_potential_preprocessing.ipynb`）：
- 重复检查（可选仅删除完全一致账户）→ 到期日筛选（可开关）→ 生效日筛选（可开关）→ 起点违约剔除
- 一客多贷聚合 → Part1 清洗
- **同时构造** `y_freq`（频繁支用，模式可调 `Y_FREQ_MODE`）与 `y_dq_risk`（违约风险）
- Part2 删除泄露字段后保存

In [ ]:
from load_kechuang_potential_data import (
    read_data,
    report_cst_loan_duplicates,
    deduplicate_exact_cst_loan,
    cast_cat_cols,
    rename_kechuang_cols,
    filter_by_maturity,
    filter_by_eff_date,
    filter_dq_start_customers,
    aggregate_by_customer_potential,
    clean_data_potential,
    build_labels_potential,
    drop_post_label_cols,
)

def _exclude_high_original_limit_customers(data, limit=MAX_ORIGINAL_CREDIT_LIMIT):
    """Keep only customers whose original credit limit is strictly below limit."""
    credit_col = "credamt" if "credamt" in data.columns else "授信额度"
    if credit_col not in data.columns:
        raise ValueError("无法剔除高额度客户：缺少 credamt/授信额度字段")
    credit = pd.to_numeric(data[credit_col], errors="coerce")
    if credit.isna().any():
        raise ValueError(f"{credit_col} 存在空值或非数值，无法可靠执行100万元门槛过滤")
    keep = credit < float(limit)
    removed = int((~keep).sum())
    print(f"  原始授信额度过滤: 剔除 >= {limit:,.0f} 元客户 {removed:,} 名；保留 {int(keep.sum()):,} 名")
    return data.loc[keep].copy()

if SKIP_CLEANING:
    print(f"⏭  SKIP_CLEANING=True，直接读取清洗后文件: {CLEANED_FILE}")
    df_cleaned = pd.read_csv(CLEANED_FILE, encoding=CSV_ENCODING)
    df_cleaned = _exclude_high_original_limit_customers(df_cleaned)
    df_cleaned.to_csv(CLEANED_FILE, index=False, encoding=CSV_ENCODING)
    print(f"  已读取并应用额度门槛: {len(df_cleaned):,} 行 × {len(df_cleaned.columns)} 列")
else:
    print("开始清洗（load_kechuang_potential_data 流程，双标签 y_freq + y_dq_risk）...")

    # 1. 读取 → 列名小写/中文恢复 → 重复检查/可选精确去重 → 类别字段转换
    df0 = read_data(DATA_FILE, csv_encoding=CSV_ENCODING)
    df0 = rename_kechuang_cols(df0)
    df0 = report_cst_loan_duplicates(df0)
    if DEDUP_CST_LOAN:
        df0 = deduplicate_exact_cst_loan(df0)
    df0 = cast_cat_cols(df0)

    # 2. 到期日筛选（开关控制）
    df0 = filter_by_maturity(
        df0,
        apply_filter=APPLY_MATURITY_FILTER,
        maturity_cutoff=MATURITY_CUTOFF,
    )

    # 3. 贷款生效日筛选（开关控制）
    if APPLY_EFF_DATE_FILTER:
        df0 = filter_by_eff_date(
            df0,
            eff_date_lower=EFF_DATE_LOWER,
            eff_date_upper=EFF_DATE_UPPER,
        )
    else:
        print("  apply_eff_date_filter=False，跳过生效日筛选")

    # 4. 剔除起点已违约客户（y_dq_risk 专用，聚合前执行）
    df0 = filter_dq_start_customers(df0)

    # 5. 一客多贷聚合
    df_agg = aggregate_by_customer_potential(df0)
    print(f"  聚合后（过滤前）: {len(df_agg):,} 客户")
    df_agg = _exclude_high_original_limit_customers(df_agg)

    # 6. Part1 清洗（target=y_dq_risk 以保留 rt_acct_stat_2_end + diff 字段供双标签构造）
    df_clean = clean_data_potential(
        df_agg, snapshot_date=SNAPSHOT_DATE, target="y_dq_risk"
    )

    # 7. 构造 y_freq：P80 等数据驱动阈值只在最早60%的标签阈值参考客户上拟合，
    #    再将固定阈值应用到验证、校准和测试集，避免标签定义看见未来分布。
    _label_dates = pd.to_datetime(df_clean['split_eff_date'], errors='coerce')
    if _label_dates.isna().any():
        raise ValueError('split_eff_date 存在空值，无法无泄漏构造 y_freq 阈值')
    _label_order = np.argsort(_label_dates.to_numpy(), kind='mergesort')
    _label_n_train = int(len(df_clean) * SPLIT_RATIOS[0])
    _label_train_mask = np.zeros(len(df_clean), dtype=bool)
    _label_train_mask[_label_order[:_label_n_train]] = True
    df_labeled, thresholds = build_labels_potential(
        df_clean,
        target="y_freq",
        y_freq_mode=Y_FREQ_MODE,
        threshold_fit_mask=_label_train_mask,
    )

    # 8. 构造 y_dq_risk（违约风险）
    df_labeled, _ = build_labels_potential(df_labeled, target="y_dq_risk")

    # 9. 同时计算两种客户支用率，并用 UTILIZATION_METHOD 选择进入优化的口径。
    # 口径1：各贷款“期末累计支用-期初累计支用”先求和，再除以客户授信额度合计。
    # 口径2：各贷款期末累计支用先求和，再除以客户授信额度合计。
    _util_den = pd.to_numeric(df_labeled["credamt"], errors="coerce")
    _util_diff_num = pd.to_numeric(
        df_labeled[UTILIZED_BALANCE_DIFF_COLUMN], errors="coerce"
    )
    if UTILIZED_BALANCE_END_COLUMN not in df0.columns:
        raise ValueError(
            f"原始账户数据缺少期末累计支用字段 {UTILIZED_BALANCE_END_COLUMN}，"
            "无法计算第二种支用率"
        )
    _end_accounts = df0[["cst_id", UTILIZED_BALANCE_END_COLUMN]].copy()
    # clean_data_potential 已把聚合后的 cst_id 转为字符串；账户明细 df0 可能仍为整数/浮点。
    # 两侧统一为去首尾空格的 pandas string，避免字符串客户号映射不到数值索引。
    _end_accounts["__cst_id_key__"] = _end_accounts["cst_id"].astype("string").str.strip()
    _labeled_cst_key = df_labeled["cst_id"].astype("string").str.strip()
    _end_accounts[UTILIZED_BALANCE_END_COLUMN] = pd.to_numeric(
        _end_accounts[UTILIZED_BALANCE_END_COLUMN], errors="coerce"
    )
    _end_sum = _end_accounts.groupby("__cst_id_key__")[
        UTILIZED_BALANCE_END_COLUMN
    ].sum(min_count=1)
    _util_end_num = _labeled_cst_key.map(_end_sum)

    _invalid_den = _util_den.isna() | (_util_den <= 0)
    _missing_diff = _util_diff_num.isna()
    _missing_end = _util_end_num.isna()
    print("  支用率字段完整性检查:")
    print(f"    授信额度缺失/非正数: {int(_invalid_den.sum()):,}")
    print(f"    差值累计支用缺失  : {int(_missing_diff.sum()):,}")
    print(f"    期末累计支用缺失  : {int(_missing_end.sum()):,}")
    print(f"    账户明细客户键数  : {int(_end_sum.index.nunique()):,}")
    print(f"    标签后客户键数    : {int(_labeled_cst_key.nunique()):,}")

    _method_missing = {"difference": _missing_diff, "end": _missing_end}
    if UTILIZATION_METHOD not in _method_missing:
        raise ValueError(
            f"UTILIZATION_METHOD={UTILIZATION_METHOD!r} 无效，只能为 'difference' 或 'end'"
        )
    _selected_invalid = _invalid_den | _method_missing[UTILIZATION_METHOD]
    if _selected_invalid.any():
        _bad_ids = df_labeled.loc[
            _selected_invalid, "cst_id"
        ].astype(str).head(10).tolist()
        raise ValueError(
            f"当前所选支用率口径 {UTILIZATION_METHOD!r} 无法计算的客户数="
            f"{int(_selected_invalid.sum())}，示例={_bad_ids}。"
            "以上完整性检查可区分授信额度异常与累计支用字段真实缺失。"
        )

    _util_lookup = df_labeled[["cst_id"]].copy()
    _util_lookup["total_credit_limit"] = _util_den.to_numpy(dtype=float)
    _util_lookup["cumulative_utilized_difference"] = _util_diff_num.to_numpy(dtype=float)
    _util_lookup["cumulative_utilized_end"] = _util_end_num.to_numpy(dtype=float)
    _util_lookup["utilization_difference_raw"] = (
        _util_lookup["cumulative_utilized_difference"] /
        _util_lookup["total_credit_limit"]
    )
    _util_lookup["utilization_end_raw"] = (
        _util_lookup["cumulative_utilized_end"] /
        _util_lookup["total_credit_limit"]
    )

    _method_to_col = {
        "difference": "utilization_difference_raw",
        "end": "utilization_end_raw",
    }
    _selected_util_col = _method_to_col[UTILIZATION_METHOD]
    _util_lookup["utilization_method"] = UTILIZATION_METHOD
    _util_lookup["utilization_raw"] = _util_lookup[_selected_util_col]
    _util_lookup["utilization_used"] = _util_lookup["utilization_raw"]
    if CAP_UTILIZATION_ABOVE_100:
        _util_lookup["utilization_used"] = _util_lookup["utilization_used"].clip(upper=1.0)
    _util_lookup["capped_above_100"] = (
        CAP_UTILIZATION_ABOVE_100 & (_util_lookup["utilization_raw"] > 1.0)
    )

    _distribution_rows = []
    _statistics_rows = []
    _quantiles = [("P95", 0.95), ("P90", 0.90), ("P70", 0.70),
                  ("P50", 0.50), ("P30", 0.30), ("P10", 0.10), ("P5", 0.05)]
    for _method, _column, _label in [
        ("difference", "utilization_difference_raw", "期初期末累计支用差值合计/授信额度"),
        ("end", "utilization_end_raw", "期末累计支用合计/授信额度"),
    ]:
        _u_all = _util_lookup[_column]
        _u = _u_all.dropna()
        if _u.empty:
            raise ValueError(f"支用率口径 {_method!r} 没有任何有效客户")
        _bands = [
            (">100%", _u > 1.0),
            ("=100%", _u.eq(1.0)),
            ("95%-<100%", (_u >= 0.95) & (_u < 1.0)),
            ("90%-<95%", (_u >= 0.90) & (_u < 0.95)),
            ("<90%", _u < 0.90),
        ]
        for _band, _mask in _bands:
            _distribution_rows.append({
                "计算口径": _method, "计算口径说明": _label, "区间": _band,
                "有效客户数": len(_u), "缺失客户数": int(_u_all.isna().sum()),
                "客户数": int(_mask.sum()), "比例": float(_mask.mean()),
                "比例_百分比": float(_mask.mean() * 100.0),
            })
        _stats = {
            "计算口径": _method,
            "计算口径说明": _label,
            "客户总数": len(_u_all),
            "有效客户数": len(_u),
            "缺失客户数": int(_u_all.isna().sum()),
            "最小值": float(_u.min()),
            "最大值": float(_u.max()),
            "平均值": float(_u.mean()),
            "中位数": float(_u.median()),
        }
        for _q_name, _q in _quantiles:
            _stats[_q_name] = float(_u.quantile(_q))
        _statistics_rows.append(_stats)

    _util_distribution = pd.DataFrame(_distribution_rows)
    _util_statistics = pd.DataFrame(_statistics_rows)
    _util_distribution["当前进入优化"] = (
        _util_distribution["计算口径"] == UTILIZATION_METHOD
    )
    _util_statistics["当前进入优化"] = (
        _util_statistics["计算口径"] == UTILIZATION_METHOD
    )
    print("\n  两种客户历史支用率区间分布（均按截断前原值统计）:")
    print(_util_distribution.to_string(index=False))
    print("\n  两种客户历史支用率描述统计与分位数:")
    print(_util_statistics.to_string(index=False))
    print(f"\n  当前进入额度优化的支用率口径: {UTILIZATION_METHOD} ({_selected_util_col})")
    print(f"  >100%截断开关: {CAP_UTILIZATION_ABOVE_100}")
    _util_lookup.to_csv(UTILIZATION_LOOKUP_FILE, index=False, encoding=CSV_ENCODING)
    _util_distribution.to_csv(
        UTILIZATION_DISTRIBUTION_FILE, index=False, encoding=CSV_ENCODING
    )
    _util_statistics.to_csv(
        UTILIZATION_STATISTICS_FILE, index=False, encoding=CSV_ENCODING
    )
    print(f"  支用率客户明细: {UTILIZATION_LOOKUP_FILE}")
    print(f"  支用率区间分布: {UTILIZATION_DISTRIBUTION_FILE}")
    print(f"  支用率统计与分位数: {UTILIZATION_STATISTICS_FILE}")

    # 10. Part2 删除泄露字段；客户支用率另存审计表，不进入概率模型特征。
    df_cleaned = drop_post_label_cols(df_labeled, target="y_dq_risk")

    # 11. 因变量分布汇报
    print(f"\n  y_freq   正例: {df_cleaned['y_freq'].mean():.2%}  "
          f"({int(df_cleaned['y_freq'].sum()):,} / {len(df_cleaned):,})")
    print(f"  y_dq_risk 正例: {df_cleaned['y_dq_risk'].mean():.2%}  "
          f"({int(df_cleaned['y_dq_risk'].sum()):,} / {len(df_cleaned):,})")
    print(f"  y_freq 模式: {thresholds.get('y_freq_mode', Y_FREQ_MODE)}")
    print("  y_freq 的 P80 阈值仅用最早60%的标签阈值参考客户拟合，并固定应用到其余样本")
    if thresholds.get("y_freq_mode") == "bout_gt0_and_curr_p80":
        print(f"    条件: ba_out_bal_diff > 0 且 ac_curr_bal_diff >= {thresholds['thr_curr']:.2f}（P80）")
    elif thresholds.get("y_freq_mode") == "curr_p80_only":
        print(f"    条件: ac_curr_bal_diff >= {thresholds['thr_curr']:.2f}（P80）")
    elif thresholds.get("y_freq_mode") == "bout_p80_and_accr_p80":
        print(f"    条件: ba_out_bal_diff >= {thresholds['thr_bout']:.2f} 且 "
              f"ac_accr_bal_diff >= {thresholds['thr_accr']:.2f}（均 P80）")
    else:
        print(f"    条件: ac_curr_bal_diff >= {thresholds['thr_curr']:.2f} 且 "
              f"ba_out_bal_diff >= {thresholds['thr_bout']:.2f}（均 P80）")

    df_cleaned.to_csv(CLEANED_FILE, index=False, encoding=CSV_ENCODING)
    print(f"\n✓ 清洗后数据已保存到: {CLEANED_FILE}")
    print(f"  shape: {df_cleaned.shape}")

df_cleaned.head(3)

---
## Cell 4 · 授信额度字段（credamt）统计

In [ ]:
# 读取聚合后数据（可能来自清洗后，也可能来自原始聚合后）
_cred_col = "credamt" if "credamt" in df_cleaned.columns else "授信额度"
credamt_series = pd.to_numeric(df_cleaned[_cred_col], errors="coerce").dropna()

cred_min  = float(credamt_series.min())
cred_max  = float(credamt_series.max())
cred_mean = float(credamt_series.mean())
cred_med  = float(credamt_series.median())

# 推算实际步长（取相邻差值的众数作为参考）
_sorted = np.sort(credamt_series.unique())
_diffs  = np.diff(_sorted)
_diffs  = _diffs[_diffs > 0]
if len(_diffs) > 0:
    _diff_mode = float(pd.Series(_diffs).mode().iloc[0])
else:
    _diff_mode = float('nan')

print("╔══════════════════════════════════════════════════════╗")
print("║         授信额度字段（credamt）统计信息               ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  字段名  : {_cred_col:<40}  ║")
print(f"║  非空样本: {len(credamt_series):>8,} 人{'':<28}║")
print(f"║  最小值  : ¥{cred_min:>12,.0f}{'':<26}║")
print(f"║  最大值  : ¥{cred_max:>12,.0f}{'':<26}║")
print(f"║  均值    : ¥{cred_mean:>12,.0f}{'':<26}║")
print(f"║  中位数  : ¥{cred_med:>12,.0f}{'':<26}║")
print(f"║  推算步长: ¥{_diff_mode:>12,.0f}（相邻唯一值差值众数）{'':<5}║")
print("╚══════════════════════════════════════════════════════╝")
print()
print(f"当前网格参数（Cell 1 设置）：")
print(f"  GRID_MIN={GRID_MIN:,.0f}, GRID_MAX={GRID_MAX:,.0f}, GRID_STEP={GRID_STEP:.0f}")
print(f"  INCLUDE_ZERO={INCLUDE_ZERO}")

# 分布直方图（简单文本版）
bins_hist = np.linspace(cred_min, cred_max, 11)
hist_vals, _ = np.histogram(credamt_series, bins=bins_hist)
print("\n分布直方图（等宽10段）：")
for lo, hi, cnt in zip(bins_hist[:-1], bins_hist[1:], hist_vals):
    bar = "█" * int(cnt / max(hist_vals) * 30)
    print(f"  [{lo/10000:5.0f}万, {hi/10000:5.0f}万) {bar} {cnt:,}")

# ── 档位字段存在性检查：确认 A/B/C 是否真的在数据中 ─────────────
print("\n档位字段存在性检查：")
_tier_col = next((c for c in ["档位", "talent_level", "level", "kechuang_level", "科创档位", "cst_star_cd"] if c in df_cleaned.columns), None)
if _tier_col is None:
    print("  未找到常见档位字段；请检查原始字段名是否变化。")
else:
    _tier_map = {
        "F3": 1, "F3级": 1, "F3档": 1, "f3": 1,
        "F2": 2, "F2级": 2, "F2档": 2, "f2": 2,
        "F1": 3, "F1级": 3, "F1档": 3, "f1": 3,
        "E": 4, "E级": 4, "E档": 4, "E类": 4, "e": 4,
        "D": 5, "D级": 5, "D档": 5, "D类": 5, "d": 5,
        "C": 6, "C级": 6, "C档": 6, "C类": 6, "c": 6,
        "B": 7, "B级": 7, "B档": 7, "B类": 7, "b": 7,
        "A": 8, "A级": 8, "A档": 8, "A类": 8, "a": 8,
    }
    _label = {1: "F3", 2: "F2", 3: "F1", 4: "E", 5: "D", 6: "C", 7: "B", 8: "A"}
    _raw_tier = df_cleaned[_tier_col]
    if pd.api.types.is_numeric_dtype(_raw_tier):
        _tier_num = pd.to_numeric(_raw_tier, errors="coerce")
    else:
        _tier_num = _raw_tier.astype(str).str.strip().map(_tier_map)
    _tier_diag = pd.DataFrame({
        "Tier": [_label[i] for i in range(1, 9)],
        "Level": list(range(1, 9)),
        "Count": [int((_tier_num == i).sum()) for i in range(1, 9)],
    })
    _tier_diag["Exists"] = _tier_diag["Count"] > 0
    print(f"  使用字段: {_tier_col}")
    print("  原始值分布（前20项）：")
    print(_raw_tier.astype(str).str.strip().value_counts(dropna=False).head(20).to_string())
    print("\n  标准化后 A~F3 分布：")
    print(_tier_diag.to_string(index=False))
    print("\n  A/B/C 是否存在：")
    print(_tier_diag[_tier_diag["Tier"].isin(["A", "B", "C"])][["Tier", "Count", "Exists"]].to_string(index=False))


---
## Cell 5 · 开发阶段概率网格与全量折外概率网格

先按 `y_freq × y_dq_risk` 联合标签分层随机划分训练集、独立概率校准集和最终测试集，与任务2使用相同随机种子和划分逻辑。类别不平衡采样只作用于模型训练集，Isotonic 只在完整校准集拟合，最终测试集不参与训练、校准或参数选择。开启额度优化抽样时，模型训练与校准范围不缩减，只为预先固定的 fit/test/all 抽样客户生成候选额度概率网格；Cell 6B/7/8 复用同一名单。随后对历史客户执行联合分层随机的外层五折交叉拟合；抽样模式下仍用完整建模折训练，只保存 all 抽样客户的折外概率网格。


In [ ]:
# 防止服务器运行到其他目录中的旧版同名脚本
import os
import json
_generator_path = os.path.abspath("generate_probability_grid.py")
if not os.path.isfile(_generator_path):
    raise FileNotFoundError(f"未找到概率网格脚本: {_generator_path}")
print("✓ 使用概率网格脚本:", _generator_path)

# 两类概率网格共享的数据、特征、采样与模型参数
CLEANED_FILE_OVERRIDE = CLEANED_FILE
EXCEL_PATH_OVERRIDE = DATA_FILE
CSV_ENCODING_OVERRIDE = CSV_ENCODING
OUT_DIR_OVERRIDE = PROB_GRID_DIR
DEV_CALIBRATED_DIR_OVERRIDE = DEV_GRID_DIR
CROSS_FIT_DIR_OVERRIDE = CROSSFIT_GRID_DIR
SPLIT_RATIOS_OVERRIDE = SPLIT_RATIOS
CROSS_FIT_FOLDS_OVERRIDE = CROSS_FIT_FOLDS
INNER_CROSS_FIT_FOLDS_OVERRIDE = INNER_CROSS_FIT_FOLDS
SNAPSHOT_DATE_OVERRIDE = SNAPSHOT_DATE
APPLY_MATURITY_FILTER_OVERRIDE = APPLY_MATURITY_FILTER
MATURITY_CUTOFF_OVERRIDE = MATURITY_CUTOFF
APPLY_EFF_DATE_FILTER_OVERRIDE = APPLY_EFF_DATE_FILTER
EFF_DATE_LOWER_OVERRIDE = EFF_DATE_LOWER
EFF_DATE_UPPER_OVERRIDE = EFF_DATE_UPPER
DEDUP_CST_LOAN_OVERRIDE = DEDUP_CST_LOAN
Y_FREQ_MODE_OVERRIDE = Y_FREQ_MODE
SAMPLING_METHOD_USAGE_OVERRIDE = SAMPLING_METHOD_USAGE
SAMPLING_STRATEGY_USAGE_OVERRIDE = SAMPLING_STRATEGY_USAGE
SAMPLING_N_ESTIMATORS_USAGE_OVERRIDE = SAMPLING_N_ESTIMATORS_USAGE
SAMPLING_ENSEMBLE_RATIO_USAGE_OVERRIDE = SAMPLING_ENSEMBLE_RATIO_USAGE
SAMPLING_METHOD_DEFAULT_OVERRIDE = SAMPLING_METHOD_DEFAULT
SAMPLING_STRATEGY_DEFAULT_OVERRIDE = SAMPLING_STRATEGY_DEFAULT
SAMPLING_N_ESTIMATORS_DEFAULT_OVERRIDE = SAMPLING_N_ESTIMATORS_DEFAULT
SAMPLING_ENSEMBLE_RATIO_DEFAULT_OVERRIDE = SAMPLING_ENSEMBLE_RATIO_DEFAULT
SAMPLING_CATEGORICAL_CANDIDATES_OVERRIDE = SAMPLING_CATEGORICAL_CANDIDATES
GRID_MIN_OVERRIDE, GRID_MAX_OVERRIDE, GRID_STEP_OVERRIDE = GRID_MIN, GRID_MAX, GRID_STEP
INCLUDE_ZERO_OVERRIDE = INCLUDE_ZERO
LGB_PARAMS_OVERRIDE = LGB_PARAMS
HANDLE_IMBALANCE_OVERRIDE = HANDLE_IMBALANCE
RANDOM_STATE_OVERRIDE = RANDOM_STATE
PROBABILITY_GRID_SAMPLE_ENABLED_OVERRIDE = OPTIMIZATION_SAMPLE_ENABLED
PROBABILITY_GRID_SAMPLE_SIZE_OVERRIDE = OPTIMIZATION_SAMPLE_SIZE
PROBABILITY_GRID_SAMPLE_RANDOM_STATE_OVERRIDE = OPTIMIZATION_SAMPLE_RANDOM_STATE

_required_grid_files = [
    "customer_id.npy", "grid.npy", "p_usage.npy", "p_default.npy",
    "p_usage_raw.npy", "p_default_raw.npy", "work_features.csv",
    "split_manifest.csv", "calibration_method.txt",
    "source_row_indices.npy", "optimization_sample_manifest.csv",
    "grid_metadata.json",
]
if REUSE_PROBABILITY_GRID:
    for _dir in (DEV_GRID_DIR, CROSSFIT_GRID_DIR):
        _needed = list(_required_grid_files)
        if _dir == DEV_GRID_DIR:
            _needed += ["full_work_features.csv", "full_fit_idx.npy"]
        _missing = [f for f in _needed if not os.path.isfile(os.path.join(_dir, f))]
        if _missing:
            raise FileNotFoundError(f"概率网格目录不完整: {_dir}, 缺少 {_missing}")
    print("⏭ 已复用开发阶段与全量折外概率网格")
else:
    # 固定采样方法后，开发模型在 train+validation 拟合；校准器只用 cal；test 最终一次评价。
    CROSS_FIT_MODE_OVERRIDE = False
    get_ipython().run_line_magic("run", '-i "{}"'.format(_generator_path))

    # 全量历史客户折外概率：外层目标折客户标签不进入对应模型或校准器。
    CROSS_FIT_MODE_OVERRIDE = True
    get_ipython().run_line_magic("run", '-i "{}"'.format(_generator_path))


---
## Cell 5B · 概率网格完整性检查

检查开发阶段 train/validation/cal/test 索引互斥且完整，并检查每位历史客户恰好获得一次折外校准预测。


In [ ]:
import os
import json
import numpy as np
import pandas as pd

_dev_required = [
    "train_idx.npy", "validation_idx.npy", "fit_idx.npy",
    "cal_idx.npy", "test_idx.npy", "split_manifest.csv",
    "customer_id.npy", "grid.npy", "p_usage.npy", "p_default.npy",
    "p_usage_raw.npy", "p_default_raw.npy", "work_features.csv",
    "full_work_features.csv", "full_fit_idx.npy",
    "source_row_indices.npy", "optimization_sample_manifest.csv",
    "grid_metadata.json",
]
_missing = [f for f in _dev_required if not os.path.isfile(os.path.join(DEV_GRID_DIR, f))]
if _missing:
    raise FileNotFoundError(f"开发阶段概率网格不完整: {_missing}")
_train_idx = np.load(os.path.join(DEV_GRID_DIR, "train_idx.npy"))
_validation_idx = np.load(os.path.join(DEV_GRID_DIR, "validation_idx.npy"))
_fit_idx = np.load(os.path.join(DEV_GRID_DIR, "fit_idx.npy"))
_cal_idx = np.load(os.path.join(DEV_GRID_DIR, "cal_idx.npy"))
_test_idx = np.load(os.path.join(DEV_GRID_DIR, "test_idx.npy"))
_dev_ids = np.load(os.path.join(DEV_GRID_DIR, "customer_id.npy"), allow_pickle=True).astype(str)
_dev_work_ids = pd.read_csv(
    os.path.join(DEV_GRID_DIR, "work_features.csv"),
    usecols=["cst_id"], encoding=CSV_ENCODING, dtype={"cst_id": str},
)["cst_id"].astype(str).to_numpy()
_dev_pu = np.load(os.path.join(DEV_GRID_DIR, "p_usage.npy"), mmap_mode="r")
_dev_pd = np.load(os.path.join(DEV_GRID_DIR, "p_default.npy"), mmap_mode="r")
_dev_sample_manifest = pd.read_csv(
    os.path.join(DEV_GRID_DIR, "optimization_sample_manifest.csv"), dtype={"cst_id": str}
)
with open(os.path.join(DEV_GRID_DIR, "grid_metadata.json"), encoding="utf-8") as _f:
    _dev_grid_meta = json.load(_f)
if not np.array_equal(_dev_ids, _dev_work_ids):
    raise ValueError("开发阶段 customer_id.npy 与 work_features.csv 客户顺序不一致")
if not np.array_equal(_dev_ids, _dev_sample_manifest["cst_id"].astype(str).to_numpy()):
    raise ValueError("开发阶段固定抽样名单与概率矩阵客户顺序不一致")
if _dev_pu.shape[0] != len(_dev_ids) or _dev_pd.shape[0] != len(_dev_ids):
    raise ValueError("开发阶段支用/违约概率网格与客户数不一致")
_split_sets = [set(x) for x in (_train_idx, _validation_idx, _cal_idx, _test_idx)]
assert all(not _split_sets[i] & _split_sets[j] for i in range(4) for j in range(i + 1, 4))
assert set(_fit_idx) == set(np.concatenate([_train_idx, _validation_idx]))
assert set(np.concatenate([_train_idx, _validation_idx, _cal_idx, _test_idx])) == set(range(len(_dev_ids)))

_cf_required = [
    "split_manifest.csv", "customer_id.npy", "grid.npy", "p_usage.npy",
    "p_default.npy", "p_usage_raw.npy", "p_default_raw.npy", "work_features.csv",
    "source_row_indices.npy", "optimization_sample_manifest.csv",
    "grid_metadata.json",
]
_missing = [f for f in _cf_required if not os.path.isfile(os.path.join(CROSSFIT_GRID_DIR, f))]
if _missing:
    raise FileNotFoundError(f"全量折外概率网格不完整: {_missing}")
_manifest = pd.read_csv(os.path.join(CROSSFIT_GRID_DIR, "split_manifest.csv"))
_cf_sample_manifest = pd.read_csv(
    os.path.join(CROSSFIT_GRID_DIR, "optimization_sample_manifest.csv"), dtype={"cst_id": str}
)
_ids = np.load(os.path.join(CROSSFIT_GRID_DIR, "customer_id.npy"), allow_pickle=True).astype(str)
_pu = np.load(os.path.join(CROSSFIT_GRID_DIR, "p_usage.npy"), mmap_mode="r")
_pd = np.load(os.path.join(CROSSFIT_GRID_DIR, "p_default.npy"), mmap_mode="r")
_grid = np.load(os.path.join(CROSSFIT_GRID_DIR, "grid.npy"))
with open(os.path.join(CROSSFIT_GRID_DIR, "grid_metadata.json"), encoding="utf-8") as _f:
    _cf_grid_meta = json.load(_f)
assert len(_manifest) == len(_ids) == _pu.shape[0] == _pd.shape[0]
if not np.array_equal(_ids, _cf_sample_manifest["cst_id"].astype(str).to_numpy()):
    raise ValueError("折外固定抽样名单与概率矩阵客户顺序不一致")
if bool(_dev_grid_meta["probability_grid_sampled"]) != bool(OPTIMIZATION_SAMPLE_ENABLED):
    raise ValueError("开发概率网格抽样口径与当前开关不一致")
if bool(_cf_grid_meta["probability_grid_sampled"]) != bool(OPTIMIZATION_SAMPLE_ENABLED):
    raise ValueError("折外概率网格抽样口径与当前开关不一致")
if OPTIMIZATION_SAMPLE_ENABLED:
    for _meta in (_dev_grid_meta, _cf_grid_meta):
        assert int(_meta["sample_size"]) == int(OPTIMIZATION_SAMPLE_SIZE)
        assert int(_meta["sample_random_state"]) == int(OPTIMIZATION_SAMPLE_RANDOM_STATE)
else:
    if not np.array_equal(_dev_ids, _ids):
        raise ValueError("全量模式下开发与折外概率网格客户池不一致，请重新运行 Cell 5")
assert _manifest["row_index"].nunique() == len(_ids)
assert set(_manifest["row_index"]) == set(range(len(_ids)))
_observed_folds = set(_manifest["fold"])
assert _observed_folds and _observed_folds.issubset(set(range(1, CROSS_FIT_FOLDS + 1)))
if not OPTIMIZATION_SAMPLE_ENABLED:
    assert _observed_folds == set(range(1, CROSS_FIT_FOLDS + 1))
assert _pu.shape[1] == _pd.shape[1] == len(_grid)
assert 0 <= float(_pu.min()) <= float(_pu.max()) <= 1
assert 0 <= float(_pd.min()) <= float(_pd.max()) <= 1
print("✓ 开发阶段数据边界与当前客户范围折外概率网格检查通过")
print("train/validation/cal/test:", len(_train_idx), len(_validation_idx), len(_cal_idx), len(_test_idx))
print(_manifest.groupby("fold").size().rename("样本数"))


---
## Cell 6 · 基于开发拟合集计算优化参数

平均支用率、线性成本和人才等级额度上限只使用 `train + validation` 客户计算。等级上限严格按照技术文档4.3.4.4计算：F3/F2/F1采用历史上限收缩，E/D分别使用1.1×U0和1.2×U0放缩，并仅要求 F3 ≤ F2 ≤ F1 ≤ E ≤ D。LGD 优先采用逐期历史回收明细；缺少可靠明细时，使用配置区明确选择的情景值并在结果中保留来源标记。


In [ ]:
import os
import pandas as pd
from parameter_selection import (
    ParameterSelectionResult, derive_tier_limit_policy,
    estimate_average_utilization, read_table, resolve_lgd, save_parameter_selection,
)

_dev_work_all = pd.read_csv(
    os.path.join(DEV_GRID_DIR, "full_work_features.csv"), encoding=CSV_ENCODING
)
_parameter_fit_idx = np.load(os.path.join(DEV_GRID_DIR, "full_fit_idx.npy"))
_development_work = _dev_work_all.iloc[_parameter_fit_idx].copy()
_talent_col = next((c for c in ("档位", "talent_level") if c in _development_work.columns), None)
if _talent_col is None:
    raise ValueError("开发拟合集缺少人才等级字段（档位/talent_level）")

# 客户支用率已在 Cell 3 按一客多贷聚合口径计算；此处只取开发拟合集做参数审计。
if not os.path.isfile(UTILIZATION_LOOKUP_FILE):
    raise FileNotFoundError(f"缺少 {UTILIZATION_LOOKUP_FILE}，请先以 SKIP_CLEANING=False 运行 Cell 3")
_util_all = pd.read_csv(
    UTILIZATION_LOOKUP_FILE, encoding=CSV_ENCODING, dtype={"cst_id": str}
)
_development_ids = set(_development_work["cst_id"].astype(str))
_util_detail = _util_all[_util_all["cst_id"].isin(_development_ids)].copy()
if len(_util_detail) != len(_development_ids):
    raise ValueError("开发拟合集与客户支用率表未完整对齐")
AVERAGE_UTILIZATION = float(_util_detail["utilization_used"].mean())
_HISTORICAL_AVERAGE_UTILIZATION = float(_util_detail["utilization_raw"].mean())
_util_summary = pd.DataFrame([{
    "customer_count": len(_util_detail),
    "raw_mean": _HISTORICAL_AVERAGE_UTILIZATION,
    "used_mean": AVERAGE_UTILIZATION,
    "raw_max": float(_util_detail["utilization_raw"].max()),
    "above_one_count": int((_util_detail["utilization_raw"] > 1.0).sum()),
    "cap_above_100_enabled": bool(CAP_UTILIZATION_ABOVE_100),
    "utilization_method": UTILIZATION_METHOD,
    "parameter_source": "customer_level_historical_utilization",
}])
if LINEAR_COST_MODE == "manual":
    LINEAR_COST = float(LINEAR_COST_MANUAL_VALUE)
elif LINEAR_COST_MODE == "ftp_utilization":
    # 仅作平均口径审计；优化器实际通过 (r-FTP)*L*u 实现逐客户 c1_i=FTP*u_i。
    LINEAR_COST = FTP_RATE * AVERAGE_UTILIZATION
else:
    raise ValueError("LINEAR_COST_MODE 只能为 'manual' 或 'ftp_utilization'")

_recovery_df = read_table(LGD_HISTORY_FILE, CSV_ENCODING) if LGD_HISTORY_FILE else None
LGD_COEFFICIENT, LGD_SOURCE, _lgd_summary, _lgd_detail = resolve_lgd(
    _recovery_df, LGD_COLUMN_MAP if _recovery_df is not None else None,
    LGD_SCENARIOS, LGD_SELECTED_SCENARIO,
)

_derived_tier_min_limits, TIER_MAX_LIMITS, _tier_summary = derive_tier_limit_policy(
    _development_work, talent_col=_talent_col, credit_limit_col="credamt",
    shrink_k=TIER_SHRINK_K, overall_quantile=TIER_OVERALL_QUANTILE,
    grid_step=GRID_STEP, grid_max=GRID_MAX,
)
TIER_MIN_LIMITS = dict(TIER_MIN_LIMIT_POLICY)
if TIER_MIN_LIMITS != _derived_tier_min_limits:
    raise RuntimeError(f'Notebook等级下限政策与参数模块不一致: notebook={TIER_MIN_LIMITS}, module={_derived_tier_min_limits}')

if not all(TIER_MAX_LIMITS[level] <= TIER_MAX_LIMITS[level + 1] for level in range(1, 5)):
    raise RuntimeError(f'等级额度上限不满足技术文档的非递减约束: {TIER_MAX_LIMITS}')

_parameter_result = ParameterSelectionResult(
    interest_rate=INTEREST_RATE, lgd_coefficient=LGD_COEFFICIENT, lgd_source=LGD_SOURCE,
    average_utilization=AVERAGE_UTILIZATION, ftp_rate=FTP_RATE, linear_cost=LINEAR_COST,
    c2_reference_limit=C2_REFERENCE_LIMIT, c2_deltas=C2_DELTAS,
    c2_candidates=C2_CANDIDATES, tier_min_limits=TIER_MIN_LIMITS,
    tier_max_limits=TIER_MAX_LIMITS, group_mean_min_ratio=GROUP_MEAN_MIN_RATIO,
    risk_tolerance=RISK_TOLERANCE,
    development_customer_count=int(_development_work["cst_id"].nunique()),
)
save_parameter_selection(
    _parameter_result, PARAMETER_REPORTS_DIR, _util_summary, _util_detail,
    _lgd_summary, _lgd_detail, _tier_summary,
)
print("✓ 参数计算完成")
print(f"  开发拟合集平均支用率（目标函数逐客户使用）={AVERAGE_UTILIZATION:.4%}; 截断前均值={_HISTORICAL_AVERAGE_UTILIZATION:.4%}")
print(f"  c1模式={LINEAR_COST_MODE}; c1参数/平均审计值={LINEAR_COST:.8f}; FTP={FTP_RATE:.2%}")
print(f"  LGD={LGD_COEFFICIENT:.4f}; 来源={LGD_SOURCE}")
print("  c2候选:", C2_CANDIDATES)
print("  等级额度上限:", TIER_MAX_LIMITS)
print("  参数审计目录:", PARAMETER_REPORTS_DIR)


---
## Cell 6B · 二次成本参数候选实测

在开发拟合集上使用当前 `SOLVER_BACKEND` 分别计算每个候选值，比较模型估计净收益、零额度率、分档上限命中率、组合风险和额度调整分布。若开启优化抽样，则直接复用 Cell 5 在开发拟合集内预先固定并生成概率网格的客户名单，不再次抽样。候选网格需要压缩时，会同时参考所有 c2 候选并构造一份公共候选额度集，每个 c2 使用完全相同的候选口径。若多个候选的净收益差异落在容忍范围内，选择较小值；最终测试集不参与选择。Python 3.6 两个近似后端比较的是同一配置下产生的可行解，不是各自的全局最优值。`lagrangian` 还会在双起点后运行限时双向局部搜索，允许单客户回退及两客户交换。


In [ ]:
EXCEL_PATH_OPT_OVERRIDE = DATA_FILE
CLEANED_FILE_OPT_OVERRIDE = CLEANED_FILE
DEDUP_CST_LOAN_OPT_OVERRIDE = DEDUP_CST_LOAN
INTEREST_RATE_OPT_OVERRIDE = INTEREST_RATE
FTP_RATE_OPT_OVERRIDE = FTP_RATE
UTILIZATION_LOOKUP_FILE_OPT_OVERRIDE = UTILIZATION_LOOKUP_FILE
LGD_COEFFICIENT_OPT_OVERRIDE = LGD_COEFFICIENT
LINEAR_COST_OPT_OVERRIDE = LINEAR_COST
LINEAR_COST_MODE_OPT_OVERRIDE = LINEAR_COST_MODE
QUADRATIC_COST_OPT_OVERRIDE = C2_CANDIDATES[0]
TOTAL_BUDGET_OPT_OVERRIDE = TOTAL_BUDGET
RISK_BUDGET_OPT_OVERRIDE = RISK_BUDGET
RISK_TOLERANCE_OPT_OVERRIDE = RISK_TOLERANCE
ENFORCE_GROUP_MEAN_MONOTONIC_OPT_OVERRIDE = ENFORCE_GROUP_MEAN_MONOTONIC
GROUP_MEAN_MIN_RATIO_OPT_OVERRIDE = GROUP_MEAN_MIN_RATIO
SOLVER_BACKEND_OPT_OVERRIDE = SOLVER_BACKEND
OPTIMIZER_MAX_VARIABLES_OPT_OVERRIDE = OPTIMIZER_MAX_VARIABLES
OPTIMIZER_CANDIDATES_PER_CUSTOMER_OPT_OVERRIDE = OPTIMIZER_CANDIDATES_PER_CUSTOMER
HEURISTIC_MAX_ROUNDS_OPT_OVERRIDE = HEURISTIC_MAX_ROUNDS
LAGRANGIAN_ITERATIONS_OPT_OVERRIDE = LAGRANGIAN_ITERATIONS
LAGRANGIAN_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LAGRANGIAN_TIME_LIMIT_SECONDS
LAGRANGIAN_STEP_SIZE_OPT_OVERRIDE = LAGRANGIAN_STEP_SIZE
LAGRANGIAN_MULTIPLIER_CAP_OPT_OVERRIDE = LAGRANGIAN_MULTIPLIER_CAP
LOCAL_SEARCH_ENABLED_OPT_OVERRIDE = LOCAL_SEARCH_ENABLED
LOCAL_SEARCH_MAX_PASSES_OPT_OVERRIDE = LOCAL_SEARCH_MAX_PASSES
LOCAL_SEARCH_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LOCAL_SEARCH_TIME_LIMIT_SECONDS
LOCAL_SEARCH_PAIR_CANDIDATE_POOL_OPT_OVERRIDE = LOCAL_SEARCH_PAIR_CANDIDATE_POOL
MILP_TIME_LIMIT_SECONDS_OPT_OVERRIDE = MILP_TIME_LIMIT_SECONDS
MILP_RELATIVE_GAP_OPT_OVERRIDE = MILP_RELATIVE_GAP
GRID_MIN_OPT_OVERRIDE, GRID_MAX_OPT_OVERRIDE, GRID_STEP_OPT_OVERRIDE = GRID_MIN, GRID_MAX, GRID_STEP
TIER_MIN_LIMITS_OPT_OVERRIDE = dict(TIER_MIN_LIMIT_POLICY)
TIER_MAX_LIMITS_OPT_OVERRIDE = TIER_MAX_LIMITS
CUSTOMER_MIN_LIMIT_ENABLED_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_ENABLED
CUSTOMER_MIN_LIMIT_DECREASE_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_DECREASE
REUSE_OPTIMIZATION_OVERRIDE = False
OPTIMIZATION_SAMPLE_ENABLED_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_ENABLED
OPTIMIZATION_SAMPLE_SIZE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_SIZE
OPTIMIZATION_SAMPLE_RANDOM_STATE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_RANDOM_STATE
PROB_GRID_DIR_OPT_OVERRIDE = DEV_GRID_DIR
REPORTS_DIR_OPT_OVERRIDE = PARAMETER_REPORTS_DIR + "/c2_selection"
OPTIMIZATION_SCOPE_OVERRIDE = "fit"
C2_SELECTION_MODE_OVERRIDE = True
C2_CANDIDATES_OPT_OVERRIDE = C2_CANDIDATES
C2_MAX_ZERO_RATE_OPT_OVERRIDE = C2_MAX_ZERO_RATE
C2_MAX_UPPER_HIT_RATE_OPT_OVERRIDE = C2_MAX_UPPER_HIT_RATE
C2_OBJECTIVE_CLOSE_TOLERANCE_OPT_OVERRIDE = C2_CLOSE_RELATIVE_TOLERANCE
globals().pop('SELECTED_QUADRATIC_COST', None)
globals().pop('C2_SENSITIVITY_SUMMARY', None)
%run -i optimal_credit_limit_precomputed_grid_large.py
audit_runtime_minimum_limit_policy(df_results, TIER_MIN_LIMIT_POLICY, CUSTOMER_MIN_LIMIT_DECREASE)

if 'SELECTED_QUADRATIC_COST' not in globals() or 'C2_SENSITIVITY_SUMMARY' not in globals():
    raise RuntimeError('Cell 6B 的 c2 候选优化未完整结束；请先查看本 Cell 上方最早出现的异常。')
QUADRATIC_COST = float(SELECTED_QUADRATIC_COST)
if FULL_DATA_C2_MODE == "manual":
    FULL_DATA_QUADRATIC_COST = float(FULL_DATA_C2_MANUAL_VALUE)
elif FULL_DATA_C2_MODE == "selected":
    FULL_DATA_QUADRATIC_COST = QUADRATIC_COST
else:
    raise ValueError("FULL_DATA_C2_MODE 只能为 'manual' 或 'selected'")
C2_SENSITIVITY_SUMMARY["reference_limit"] = C2_REFERENCE_LIMIT
C2_SENSITIVITY_SUMMARY["implied_delta_at_reference"] = (
    C2_SENSITIVITY_SUMMARY["c2"] * C2_REFERENCE_LIMIT
)
C2_SENSITIVITY_SUMMARY.to_csv(
    os.path.join(PARAMETER_REPORTS_DIR, "c2_selection", "c2_sensitivity_summary.csv"),
    index=False, encoding=CSV_ENCODING,
)
import json
_derived_path = os.path.join(PARAMETER_REPORTS_DIR, "derived_parameters.json")
with open(_derived_path, "r", encoding="utf-8") as _f:
    _derived_parameters = json.load(_f)
_derived_parameters["selected_c2"] = QUADRATIC_COST
_derived_parameters["full_data_c2_mode"] = FULL_DATA_C2_MODE
_derived_parameters["full_data_c2"] = FULL_DATA_QUADRATIC_COST
_derived_parameters["c2_selection_report"] = os.path.join(
    PARAMETER_REPORTS_DIR, "c2_selection", "c2_sensitivity_summary.csv"
)
with open(_derived_path, "w", encoding="utf-8") as _f:
    json.dump(_derived_parameters, _f, ensure_ascii=False, indent=2)
print(f"✓ 开发拟合集候选选择 c2={QUADRATIC_COST:.12g}")
print(f"✓ 全量数据应用 c2={FULL_DATA_QUADRATIC_COST:.12g}（模式={FULL_DATA_C2_MODE}）")
display(C2_SENSITIVITY_SUMMARY)


---
## Cell 7 · 最终测试集额度优化离线评价

仅对最终测试客户使用开发阶段模型和独立校准器生成的概率进行一次联合优化；开启优化抽样时，直接复用 Cell 5 在测试客户范围内预先固定并生成概率网格的名单。默认总额度预算等于实际进入本次优化客户的历史总额度，风险预算等于同一客户范围历史额度加权校准违约风险的 105%；测试集不用于调参。`lagrangian` 与 `heuristic` 都会检查全部硬约束，但只给出近似可行解，不保证整数规划全局最优，也不报告 MIP gap。


In [ ]:
EXCEL_PATH_OPT_OVERRIDE = DATA_FILE
CLEANED_FILE_OPT_OVERRIDE = CLEANED_FILE
DEDUP_CST_LOAN_OPT_OVERRIDE = DEDUP_CST_LOAN
INTEREST_RATE_OPT_OVERRIDE = INTEREST_RATE
FTP_RATE_OPT_OVERRIDE = FTP_RATE
UTILIZATION_LOOKUP_FILE_OPT_OVERRIDE = UTILIZATION_LOOKUP_FILE
LGD_COEFFICIENT_OPT_OVERRIDE = LGD_COEFFICIENT
LINEAR_COST_OPT_OVERRIDE = LINEAR_COST
LINEAR_COST_MODE_OPT_OVERRIDE = LINEAR_COST_MODE
QUADRATIC_COST_OPT_OVERRIDE = QUADRATIC_COST
TOTAL_BUDGET_OPT_OVERRIDE = TOTAL_BUDGET
RISK_BUDGET_OPT_OVERRIDE = RISK_BUDGET
RISK_TOLERANCE_OPT_OVERRIDE = RISK_TOLERANCE
ENFORCE_GROUP_MEAN_MONOTONIC_OPT_OVERRIDE = ENFORCE_GROUP_MEAN_MONOTONIC
GROUP_MEAN_MIN_RATIO_OPT_OVERRIDE = GROUP_MEAN_MIN_RATIO
SOLVER_BACKEND_OPT_OVERRIDE = SOLVER_BACKEND
OPTIMIZER_MAX_VARIABLES_OPT_OVERRIDE = OPTIMIZER_MAX_VARIABLES
OPTIMIZER_CANDIDATES_PER_CUSTOMER_OPT_OVERRIDE = OPTIMIZER_CANDIDATES_PER_CUSTOMER
HEURISTIC_MAX_ROUNDS_OPT_OVERRIDE = HEURISTIC_MAX_ROUNDS
LAGRANGIAN_ITERATIONS_OPT_OVERRIDE = LAGRANGIAN_ITERATIONS
LAGRANGIAN_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LAGRANGIAN_TIME_LIMIT_SECONDS
LAGRANGIAN_STEP_SIZE_OPT_OVERRIDE = LAGRANGIAN_STEP_SIZE
LAGRANGIAN_MULTIPLIER_CAP_OPT_OVERRIDE = LAGRANGIAN_MULTIPLIER_CAP
LOCAL_SEARCH_ENABLED_OPT_OVERRIDE = LOCAL_SEARCH_ENABLED
LOCAL_SEARCH_MAX_PASSES_OPT_OVERRIDE = LOCAL_SEARCH_MAX_PASSES
LOCAL_SEARCH_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LOCAL_SEARCH_TIME_LIMIT_SECONDS
LOCAL_SEARCH_PAIR_CANDIDATE_POOL_OPT_OVERRIDE = LOCAL_SEARCH_PAIR_CANDIDATE_POOL
MILP_TIME_LIMIT_SECONDS_OPT_OVERRIDE = MILP_TIME_LIMIT_SECONDS
MILP_RELATIVE_GAP_OPT_OVERRIDE = MILP_RELATIVE_GAP
GRID_MIN_OPT_OVERRIDE, GRID_MAX_OPT_OVERRIDE, GRID_STEP_OPT_OVERRIDE = GRID_MIN, GRID_MAX, GRID_STEP
TIER_MIN_LIMITS_OPT_OVERRIDE = dict(TIER_MIN_LIMIT_POLICY)
TIER_MAX_LIMITS_OPT_OVERRIDE = TIER_MAX_LIMITS
CUSTOMER_MIN_LIMIT_ENABLED_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_ENABLED
CUSTOMER_MIN_LIMIT_DECREASE_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_DECREASE
REUSE_OPTIMIZATION_OVERRIDE = REUSE_OPTIMIZATION
OPTIMIZATION_SAMPLE_ENABLED_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_ENABLED
OPTIMIZATION_SAMPLE_SIZE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_SIZE
OPTIMIZATION_SAMPLE_RANDOM_STATE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_RANDOM_STATE
C2_SELECTION_MODE_OVERRIDE = False
PROB_GRID_DIR_OPT_OVERRIDE = DEV_GRID_DIR
REPORTS_DIR_OPT_OVERRIDE = TEST_REPORTS_DIR
OPTIMIZATION_SCOPE_OVERRIDE = "test"
%run -i optimal_credit_limit_precomputed_grid_large.py
audit_runtime_minimum_limit_policy(df_results, TIER_MIN_LIMIT_POLICY, CUSTOMER_MIN_LIMIT_DECREASE)

test_calculator = calculator
test_talent_levels = talent_levels
test_df_results = df_results
print("✓ 最终测试集离线评价完成，结果目录:", TEST_REPORTS_DIR)


---
## Cell 8 · 全部历史客户折外组合优化

使用外层交叉拟合产生的样本外校准概率执行联合离散组合优化。关闭优化抽样时覆盖所有历史客户；开启时直接复用 Cell 5 预先固定并生成折外概率网格的 all 范围抽样名单。总额度预算、风险预算和人才等级平均额度约束同时作用于实际进入本次优化的客户组合，不能逐客户分别取最优额度。两个 Python 3.6 后端都返回经过硬约束审计的近似可行解；`lagrangian` 会自动保留贪心保底解，不应表述为全局最优解。


In [ ]:
EXCEL_PATH_OPT_OVERRIDE = DATA_FILE
CLEANED_FILE_OPT_OVERRIDE = CLEANED_FILE
DEDUP_CST_LOAN_OPT_OVERRIDE = DEDUP_CST_LOAN
INTEREST_RATE_OPT_OVERRIDE = INTEREST_RATE
FTP_RATE_OPT_OVERRIDE = FTP_RATE
UTILIZATION_LOOKUP_FILE_OPT_OVERRIDE = UTILIZATION_LOOKUP_FILE
LGD_COEFFICIENT_OPT_OVERRIDE = LGD_COEFFICIENT
LINEAR_COST_OPT_OVERRIDE = LINEAR_COST
LINEAR_COST_MODE_OPT_OVERRIDE = LINEAR_COST_MODE
QUADRATIC_COST_OPT_OVERRIDE = FULL_DATA_QUADRATIC_COST
TOTAL_BUDGET_OPT_OVERRIDE = TOTAL_BUDGET
RISK_BUDGET_OPT_OVERRIDE = RISK_BUDGET
RISK_TOLERANCE_OPT_OVERRIDE = RISK_TOLERANCE
ENFORCE_GROUP_MEAN_MONOTONIC_OPT_OVERRIDE = ENFORCE_GROUP_MEAN_MONOTONIC
GROUP_MEAN_MIN_RATIO_OPT_OVERRIDE = GROUP_MEAN_MIN_RATIO
SOLVER_BACKEND_OPT_OVERRIDE = SOLVER_BACKEND
OPTIMIZER_MAX_VARIABLES_OPT_OVERRIDE = OPTIMIZER_MAX_VARIABLES
OPTIMIZER_CANDIDATES_PER_CUSTOMER_OPT_OVERRIDE = OPTIMIZER_CANDIDATES_PER_CUSTOMER
HEURISTIC_MAX_ROUNDS_OPT_OVERRIDE = HEURISTIC_MAX_ROUNDS
LAGRANGIAN_ITERATIONS_OPT_OVERRIDE = LAGRANGIAN_ITERATIONS
LAGRANGIAN_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LAGRANGIAN_TIME_LIMIT_SECONDS
LAGRANGIAN_STEP_SIZE_OPT_OVERRIDE = LAGRANGIAN_STEP_SIZE
LAGRANGIAN_MULTIPLIER_CAP_OPT_OVERRIDE = LAGRANGIAN_MULTIPLIER_CAP
LOCAL_SEARCH_ENABLED_OPT_OVERRIDE = LOCAL_SEARCH_ENABLED
LOCAL_SEARCH_MAX_PASSES_OPT_OVERRIDE = LOCAL_SEARCH_MAX_PASSES
LOCAL_SEARCH_TIME_LIMIT_SECONDS_OPT_OVERRIDE = LOCAL_SEARCH_TIME_LIMIT_SECONDS
LOCAL_SEARCH_PAIR_CANDIDATE_POOL_OPT_OVERRIDE = LOCAL_SEARCH_PAIR_CANDIDATE_POOL
MILP_TIME_LIMIT_SECONDS_OPT_OVERRIDE = MILP_TIME_LIMIT_SECONDS
MILP_RELATIVE_GAP_OPT_OVERRIDE = MILP_RELATIVE_GAP
GRID_MIN_OPT_OVERRIDE, GRID_MAX_OPT_OVERRIDE, GRID_STEP_OPT_OVERRIDE = GRID_MIN, GRID_MAX, GRID_STEP
TIER_MIN_LIMITS_OPT_OVERRIDE = dict(TIER_MIN_LIMIT_POLICY)
TIER_MAX_LIMITS_OPT_OVERRIDE = TIER_MAX_LIMITS
CUSTOMER_MIN_LIMIT_ENABLED_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_ENABLED
CUSTOMER_MIN_LIMIT_DECREASE_OPT_OVERRIDE = CUSTOMER_MIN_LIMIT_DECREASE
REUSE_OPTIMIZATION_OVERRIDE = REUSE_OPTIMIZATION
OPTIMIZATION_SAMPLE_ENABLED_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_ENABLED
OPTIMIZATION_SAMPLE_SIZE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_SIZE
OPTIMIZATION_SAMPLE_RANDOM_STATE_OPT_OVERRIDE = OPTIMIZATION_SAMPLE_RANDOM_STATE
C2_SELECTION_MODE_OVERRIDE = False
PROB_GRID_DIR_OPT_OVERRIDE = CROSSFIT_GRID_DIR
REPORTS_DIR_OPT_OVERRIDE = FULL_REPORTS_DIR
OPTIMIZATION_SCOPE_OVERRIDE = "all"
%run -i optimal_credit_limit_precomputed_grid_large.py
audit_runtime_minimum_limit_policy(df_results, TIER_MIN_LIMIT_POLICY, CUSTOMER_MIN_LIMIT_DECREASE)

full_calculator = calculator
full_talent_levels = talent_levels
full_df_results = df_results
# 后续原有诊断代码继续使用这三个兼容变量。
calculator, talent_levels, df_results = full_calculator, full_talent_levels, full_df_results
print("✓ 全量历史客户折外组合优化完成，结果目录:", FULL_REPORTS_DIR)


---
## Cell 8B · 额度来源诊断

在全量历史客户折外组合优化完成后、可视化前，检查清洗数据、概率网格和优化结果中的额度上限与 110 万以上额度来源。


In [ ]:
import os
import numpy as np
import pandas as pd

print("=" * 80)
print("额度来源诊断")
print("=" * 80)

# 1. 原始清洗数据
raw = pd.read_csv(CLEANED_FILE, encoding="utf-8-sig")
cred_col = "credamt" if "credamt" in raw.columns else "授信额度"

print("\n[1] 清洗数据")
print("字段:", cred_col)
print("max:", raw[cred_col].max())
print(">=110万:", (raw[cred_col] >= 1_100_000).sum())
print("最大20条:")
print(raw[cred_col].nlargest(20).to_string(index=False))

# 2. 概率网格
grid_dir = CROSSFIT_GRID_DIR
if not os.path.exists(os.path.join(grid_dir, "grid.npy")):
    grid_dir = PROB_GRID_DIR + "_calibrated"
if not os.path.exists(os.path.join(grid_dir, "grid.npy")):
    grid_dir = PROB_GRID_DIR

grid = np.load(os.path.join(grid_dir, "grid.npy"))

print("\n[2] 优化概率网格")
print("目录:", grid_dir)
print("grid min:", grid.min())
print("grid max:", grid.max())
print("grid点数:", len(grid))

# 3. 优化结果CSV
result_path = os.path.join(FULL_REPORTS_DIR, "credit_limit_large_grid_results.csv")
res = pd.read_csv(result_path)

print("\n[3] 优化结果CSV")
print("original_credit_limit max:",
      res["original_credit_limit"].max())
print("credit_limit max:",
      res["credit_limit"].max())
print("original >=110万:",
      (res["original_credit_limit"] >= 1_100_000).sum())
print("optimized >=110万:",
      (res["credit_limit"] >= 1_100_000).sum())


---
## Cell 8C · 五折 OOF 与组合约束诊断

直接复用 Cell 5 保存的五折 OOF 概率网格和 全量折外优化保留的优化器状态，依次生成：
1. 原始额度分箱下的原始概率、校准概率、实际正样本率和样本数；
2. 第一折快速 ALE 与五折平均 ALE；
3. 候选额度概率曲线的单调性、方向、首尾变化及代表性轨迹；
4. 同一风险调整目标下逐客独立最优额度与联合组合约束最终额度的对照。

结果保存到全量折外优化目录下的 `oof_diagnostics/`。本 Cell 不重新训练模型。

In [ ]:
OOF_ANALYSIS_GRID_DIR_OVERRIDE = CROSSFIT_GRID_DIR
OOF_ANALYSIS_OUT_DIR_OVERRIDE = f'{FULL_REPORTS_DIR}/oof_diagnostics'
OOF_BIN_COUNT_OVERRIDE = 10
ALE_BIN_COUNT_OVERRIDE = 20
CURVE_TOLERANCE_OVERRIDE = 1e-7
%run -i analyze_oof_credit_limit.py


---
## Cell 9 · 可视化图表
运行 `viz_credit_limit.py`，生成全部图表到全量折外优化目录下的 `viz/`。利润对比图直接读取优化器按本次真实运行参数保存的优化前后目标函数值，不在可视化中重新代入默认参数。

如需修改图表样式，只需编辑 `viz_credit_limit.py`，再重新运行本 Cell 即可。

In [ ]:
# 把可视化参数传给 viz_credit_limit.py
RESULTS_CSV_OVERRIDE              = f"{FULL_REPORTS_DIR}/credit_limit_large_grid_results.csv"
TALENT_NPY_OVERRIDE               = f"{FULL_REPORTS_DIR}/talent_levels.npy"
PROB_GRID_DIR_OVERRIDE            = PROB_GRID_DIR
# 校准概率网格目录：viz 的概率查表也用校准后概率，与 Fig 5 表格保持一致
PROB_GRID_DIR_CALIBRATED_OVERRIDE = CROSSFIT_GRID_DIR
VIZ_OUT_DIR_OVERRIDE              = VIZ_DIR

%run -i viz_credit_limit.py

# ── 在 notebook 中内嵌显示图表 ──
from IPython.display import Image, display
import glob
for img_path in sorted(glob.glob(f"{VIZ_DIR}/*.png")):
    print(f"\n{os.path.basename(img_path)}")
    display(Image(img_path, width=900))


---
## Cell 10 · 典型客户评分卡（暂停）

交叉拟合会产生5组折模型，不再存在一个可以代表全量客户的单一 LightGBM 模型。原评分卡脚本假定单一全量模型，若继续运行会与额度优化概率口径不一致，因此在本应用流程中暂停。


In [ ]:
print("交叉拟合应用模式下已跳过单模型 TreeSHAP 评分卡。")
